In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip uninstall -y datasets

Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5


In [3]:
!pip install datasets==2.17

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf

In [4]:
import json
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from transformers import AlbertTokenizerFast
from transformers import AlbertForTokenClassification
from sklearn.metrics import accuracy_score, classification_report
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [5]:
# Load dataset
dataset_restaurant = load_dataset("jakartaresearch/semeval-absa", name='restaurant')
train_ds = dataset_restaurant["train"]
test_ds = dataset_restaurant["validation"]

Generating train split:   0%|          | 0/3044 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/800 [00:00<?, ? examples/s]

In [6]:
# Label map
label_map = {"O": 0, "B": 1, "I": 2}
id2label = {v: k for k, v in label_map.items()}

In [7]:
#Load bert model
model = AlbertForTokenClassification.from_pretrained("albert-base-v2", num_labels=len(label_map))

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/23 [00:00<?, ?it/s]

AlbertForTokenClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
albert.pooler.weight         | UNEXPECTED | 
albert.pooler.bias           | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
#Load bert tokenizer
tokenizer = AlbertTokenizerFast.from_pretrained("albert-base-v2")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# Apply PEFT with LoRA
lora_config = LoraConfig(
    r=8,  # Rank of the LoRA matrix
    lora_alpha=16,  # Scaling factor
    target_modules= ["attention.query", "attention.key", "attention.value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.TOKEN_CLS  # Assuming token classification task
)
peft_model = get_peft_model(model, lora_config)
peft_model.train()

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


PeftModelForTokenClassification(
  (base_model): LoraModel(
    (model): AlbertForTokenClassification(
      (albert): AlbertModel(
        (embeddings): AlbertEmbeddings(
          (word_embeddings): Embedding(30000, 128, padding_idx=0)
          (position_embeddings): Embedding(512, 128)
          (token_type_embeddings): Embedding(2, 128)
          (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0, inplace=False)
        )
        (encoder): AlbertTransformer(
          (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
          (albert_layer_groups): ModuleList(
            (0): AlbertLayerGroup(
              (albert_layers): ModuleList(
                (0): AlbertLayer(
                  (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
                  (attention): AlbertAttention(
                    (attention_dropout): Dropout(p=0, inplace=False)
                 

In [10]:
# Tokenize
def tokenize_and_align(ex):
    text = ex["text"]

    terms = ex["aspects"]["term"]
    from_idx = ex["aspects"]["from"]
    to_idx = ex["aspects"]["to"]

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    offsets = encoding["offset_mapping"][0]

    label_ids = [-100] * len(offsets)

    # assign O to all valid non-padding tokens
    for i, (s, e) in enumerate(offsets):
        if s == e:
            continue
        label_ids[i] = label_map["O"]

    # assign BIO labels
    for start, end in zip(from_idx, to_idx):

        span_tokens = []

        for i, (s, e) in enumerate(offsets):

            if s == e:
                continue

            # overlap-based matching
            if not (e <= start or s >= end):
                span_tokens.append(i)

        if len(span_tokens) > 0:

            label_ids[span_tokens[0]] = label_map["B"]

            for idx in span_tokens[1:]:
                label_ids[idx] = label_map["I"]

    return (
        encoding["input_ids"].squeeze(),
        encoding["attention_mask"].squeeze(),
        torch.tensor(label_ids)
    )

In [11]:
# Build tensors
def prepare(ds):
    input_ids, attention_masks, label_ids = [], [], []
    for ex in ds:
        a, b, c = tokenize_and_align(ex)
        input_ids.append(a)
        attention_masks.append(b)
        label_ids.append(c)
    return TensorDataset(torch.stack(input_ids), torch.stack(attention_masks), torch.stack(label_ids))



In [12]:
train_dataset = prepare(train_ds)
test_dataset = prepare(test_ds)

In [13]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=5e-5)

In [14]:
# Training loop
for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = batch
        outputs = peft_model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"Epoch {epoch + 1} loss: {loss.item()}")

Epoch 1 loss: 0.0840378925204277
Epoch 2 loss: 0.2981990873813629
Epoch 3 loss: 0.0663057416677475


In [15]:
# Test data
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)
peft_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = batch
        logits = peft_model(input_ids, attention_mask=attention_mask).logits
        preds = torch.argmax(logits, dim=2)

        preds = preds.flatten()
        labels = labels.flatten()

        for p, l in zip(preds, labels):

            if l.item() == -100:
                continue

            all_preds.append(id2label[p.item()])
            all_labels.append(id2label[l.item()])

In [16]:
aspects = []
current = []

for token, label in zip(all_preds, all_labels):
    if label == "B":
        if current:
            aspects.append(" ".join(current))
        current = [token]
    elif label == "I" and current:
        current.append(token)
    elif label == "O" and current:
        aspects.append(" ".join(current))
        current = []

if current:
    aspects.append(" ".join(current))

print("Extracted Aspects:", aspects)
report = classification_report(
    all_labels,
    all_preds,
    digits=4,
    output_dict=True
)

print("\n===== OVERALL RESULTS =====")

print(
    f"Accuracy: "
    f"{accuracy_score(all_labels, all_preds):.4f}"
)

print(
    f"Macro Precision: "
    f"{report['macro avg']['precision']:.4f}"
)

print(
    f"Macro Recall: "
    f"{report['macro avg']['recall']:.4f}"
)

print(
    f"Macro F1: "
    f"{report['macro avg']['f1-score']:.4f}"
)

print(
    f"Weighted Precision: "
    f"{report['weighted avg']['precision']:.4f}"
)

print(
    f"Weighted Recall: "
    f"{report['weighted avg']['recall']:.4f}"
)

print(
    f"Weighted F1: "
    f"{report['weighted avg']['f1-score']:.4f}"
)

print("\n===== CLASS REPORT =====")

print(
    classification_report(
        all_labels,
        all_preds,
        digits=4
    )
)

Extracted Aspects: ['B', 'B O', 'B', 'B', 'B I I', 'B', 'O', 'B I', 'B I I', 'B I', 'B I', 'B', 'B', 'I I I I I', 'B', 'B I I', 'B', 'B', 'B I', 'B', 'B I', 'B', 'B I', 'B', 'B I I', 'B', 'B', 'B', 'B', 'B I O B', 'B O I', 'B I O O O', 'B', 'B I', 'B I', 'B I I O O', 'B', 'B', 'B I', 'B I B I I', 'B', 'B', 'B', 'B I', 'B I I', 'B', 'B', 'B', 'B', 'B I I I', 'B I', 'B', 'B', 'B', 'B I', 'B I', 'B', 'B I I I', 'B I', 'B I I I I I I I', 'B', 'B I I', 'B', 'B', 'B O', 'B I I I I I I I I I I I I I', 'B I I I', 'B I', 'B I', 'B I I I I', 'B I I I', 'B', 'B I I I', 'B', 'B I I I', 'B I I I I', 'B', 'B', 'B', 'B', 'B', 'B', 'B I', 'B', 'B', 'B', 'B I I I', 'B', 'B', 'O O', 'B', 'B', 'B', 'B I I', 'B', 'B I I', 'B I I', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B I I', 'B I I', 'B', 'B', 'I', 'B', 'B', 'B I I I I', 'B', 'B', 'B', 'B', 'B', 'B I I', 'B', 'B I', 'B', 'B I I', 'B I', 'B I', 'B I I', 'B', 'B', 'B I', 'B I', 'B I I I', 'B', 'B', 'B', 'B', 'B I I', 'B I', 'B I I I', 'B', 'B', 'I'

In [17]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    all_labels,
    all_preds,
    labels=["O", "B", "I"]
)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[11784    91    92]
 [   96  1016    22]
 [  107    44  1033]]
